# Laminar Counter-flow Flame

## Preamble

In [1]:
# Standard Library
from pathlib import Path
import foamnordic as fno

### Directories

In [2]:
# Case Type and Name
CASE_TYPE = "laminar"
CASE_NAME = "counterFlowFlame2D"
CATEGORY = "combustion"

# FoamNordic Project Directory
PROJECT_DIR = Path.cwd().resolve()
BASE_DIR = next((path for path in (PROJECT_DIR, *PROJECT_DIR.parents) if (path / "tutorials" / "openfoam_tutorials").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run this notebook from inside the FoamNordic source tree")
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / CATEGORY
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / CATEGORY / CASE_TYPE / CASE_NAME

# Output Directory
OUTPUT_DIR = MAIN_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# HPC configuration
ACCOUNT = "<allocation-account>" # e.g. project_xxxxxxx
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "16G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=None,
)

## Example - OpenFOAM on FoamNordic (Baseline)

### Case Definition

In [4]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="reactingFoam",
    ranks=N_TASKS,
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [5]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(
    case=case,
    scheduler=scheduler,
    verbose=True,
)

Resource,OpenFOAM,Model,Allocation
Processes,1 tasks,0 host(s),1 node(s)
CPU,1 cores,0 cores,1 cores
Memory,16 GiB,0 KiB,16 GiB
Field programs,-,0,0
SHM,-,none,none


In [6]:
# Launch the workload
run = longship.launch(verbose=None)

[FoamNordic] Preparing mesh with blockMesh: counterFlowFlame2D
[FoamNordic] Mesh is ready: counterFlowFlame2D
[FoamNordic] Sailing submitted with Job ID: 1290333
[FoamNordic] Sailing has launched with Job ID: 1290333
[FoamNordic] Sailing started at: 2026-09-13T15:41:48
[FoamNordic] Sailing in background: counterFlowFlame2D


In [7]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [8]:
# Summary of the job result
result.summary(style="compact", display=True);

Job ID,Name,Status,Partition,Node,Elapsed
1290333,counterFlowFlame2D,succeeded,small,rc4142,00:00:36


### Post-processing

In [9]:
# Postprocessing
post = result.postprocess

velocity = post.field(name="U", time_idx=-1, physical_time=None)
pressure = post.field(name="p", time_idx=-1, physical_time=None)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    fields=["U", "p"],
    time_idx=-1,
    physical_time=None,
    verbose=True,
)

U shape: (4000, 3)
p shape: (4000,)


Field,Min,Max,Mean,Std,RMS
U,9.681061e-03,3.676852e-01,1.597420e-01,8.080338e-02,1.790159e-01
p,1.000000e+05,1.000000e+05,1.000000e+05,0.000000e+00,1.000000e+05
